# 02. Models and clean performance

The model families and how well they detect fraud without an attacker: Table S-II (hyperparameter trials,
`reports/tables/hpo_trials.csv`) and the clean performance of Section VI-A and Table S-IV
(`reports/table1_clean.csv`).

**Inputs:** `runs/<dataset>/<family>/best_params.json` and `hpo_v2.json` (the hyperparameter searches) and
`metrics.json` (test metrics of each seed), written by `fraudadv.hpo.train_all` and `fraudadv.hpo.train_context`;
the trained models `runs/ulb/<family>/seed<k>.pkl` for the live check.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

## Table S-II: hyperparameter trials

`tables.hpo_trials`: the number of finished trials of the search that produced the final model of each family on
each dataset (`best_params.json`, or `hpo_v2.json` where a second tuning pass ran). The families themselves are in
`src/fraudadv/models/` (`zoo.py`, `graph.py`, `foundation.py`).

In [ ]:
from fraudadv.report import tables

trials = tables.hpo_trials()
trials.pivot(index="model", columns="dataset", values="n_trials")

## Clean performance (`reports/table1_clean.csv`, Section VI-A and Table S-IV)

`make_report.table_clean` averages the test metrics of the three seeds of every model (AUC, KS, AUPRC, recall at
1% FPR, unit- and dollar-weighted) into `reports/table1_clean.{csv,md}`: the mean $\pm$ standard deviation columns
are Table S-IV, and the `_<metric>` columns hold the numeric means quoted in Section VI-A.

In [ ]:
from fraudadv.report import make_report as MR

MR.REP.mkdir(exist_ok=True)          # make_report.main creates reports/; table_clean alone does not
MR.table_clean()
clean = nb.show_csv("reports/table1_clean.csv", rows=12)
print(len(clean), "rows (dataset x model)")

## Run it yourself: test AUC of the saved ULB models

Load the saved XGBoost and MLP of ULB (all three seeds) with `fraudadv.attacks.run_attacks.load_model`, the way the
attacks load them, score the test split and compute the weighted test AUC with `fraudadv.eval.metrics.auc`. The
mean over seeds is compared with the `_auc` column of `reports/table1_clean.csv`. This recomputes the study's
own numbers (whole test split); nothing is written to disk.

In [ ]:
import numpy as np
import pandas as pd
from fraudadv.attacks.run_attacks import load_model
from fraudadv.data.load import load
from fraudadv.eval.metrics import auc

a, meta = load("ulb")
table = pd.read_csv("reports/table1_clean.csv").set_index(["Dataset", "Model"])
rows = []
for fam in ("xgboost", "mlp"):
    aucs = [auc(a["y_test"], load_model("ulb", fam, seed).logit(a["X_test"]), a["w_test"]) for seed in (0, 1, 2)]
    stored = table.loc[("ULB", MR.FAM_NAME[fam]), "_auc"]
    rows.append(dict(model=MR.FAM_NAME[fam], **{f"auc seed {s}": v for s, v in enumerate(aucs)},
                     live_mean=np.mean(aucs), table1_clean=stored, abs_diff=abs(np.mean(aucs) - stored)))
live = pd.DataFrame(rows)
display(live)
assert (live.abs_diff < 1e-9).all()
print("live test AUC matches reports/table1_clean.csv")

## Rerun the runs behind these results from scratch

The training step of `scripts/reproduce.sh <dataset> --models train --stages models`: hyperparameter search and
three seeds of every tabular family (`python -m fraudadv.hpo.train_all <dataset> --families ...`), and on the
datasets with histories the graph and foundation models (`python -m fraudadv.hpo.train_context <dataset>
--families hgnn,hgt,chronos2,timesfm`). It takes hours (days on the large datasets) and writes into
`runs/<dataset>/<family>/`. Not executed here.

In [ ]:
RERUN = False
DATASET = "ulb"
if RERUN:
    import optuna
    from fraudadv.hpo import train_all, train_context

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    train_all.run(DATASET, ["xgboost", "catboost", "mlp", "ft_transformer", "ivis"])
    if DATASET in ("sparkov", "tabformer", "ieee_cis", "paysim"):          # card/account histories
        train_context.run(DATASET, ["hgnn", "hgt", "chronos2", "timesfm"])

In [ ]:
ok = nb.check(["reports/tables/hpo_trials.csv", "reports/table1_clean.csv", "reports/table1_clean.md"])